# Classic 

In [1]:
from pyiron_atomistics import Project

In [2]:
pr = Project("test")

In [3]:
pr.remove_jobs(recursive=True, silently=True)

0it [00:00, ?it/s]

In [4]:
structure = pr.create.structure.ase.bulk("Al", cubic=True)

In [5]:
job = pr.create.job.Lammps("lmp")

In [6]:
job.structure = structure

In [7]:
job.calc_md(pressure=0.0, temperature=500.0, seed=80996)

In [8]:
job.run()

/srv/conda/envs/notebook/lib/python3.13/site-packages/pyiron_atomistics/lammps/base.py:284: UserWarning: No potential set via job.potential - use default potential, 1995--Angelo-J-E--Ni-Al-H--LAMMPS--ipr1
  warnings.warn(


The job lmp was saved and received the ID: 1


In [9]:
job.decompress()

In [10]:
job["log.lammps"]

/tmp/ipykernel_145/1290020846.py:1: DeprecationWarning: pyiron_base.jobs.job.base.__getitem__ is deprecated: Use job.output for results, job.files to access files; job.content to access HDF storage and job.child_project to access children of master jobs..
  job["log.lammps"]


['LAMMPS (29 Aug 2024)\n',
 'OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)\n',
 '  using 1 OpenMP thread(s) per MPI task\n',
 'units metal\n',
 'dimension 3\n',
 'boundary p p p\n',
 'atom_style atomic\n',
 'read_data structure.inp\n',
 'Reading data file ...\n',
 '  orthogonal box = (0 0 0) to (4.05 4.05 4.05)\n',
 '  1 by 1 by 1 MPI processor grid\n',
 '  reading atoms ...\n',
 '  4 atoms\n',
 '  read_data CPU = 0.001 seconds\n',
 'include potential.inp\n',
 'pair_style eam/alloy\n',
 'pair_coeff * * NiAlH_jea.eam.alloy Ni Al H\n',
 'fix ensemble all npt temp 500.0 500.0 0.1 iso 0.0 0.0 1.0\n',
 'variable dumptime  equal 100\n',
 'variable thermotime  equal 100\n',
 'timestep 0.001\n',
 'velocity all create 1000.0 80996 dist gaussian\n',
 'dump 1 all custom ${dumptime} dump.out id type xsu ysu zsu fx fy fz vx vy vz\n',
 'dump 1 all custom 100 dump.out id type xsu ysu zsu fx fy fz vx vy vz\n',
 'dump_modify 1 sort id format line "%d %d %20.15g %20.1

In [11]:
job["output/generic/temperature"]

/tmp/ipykernel_145/3340484072.py:1: DeprecationWarning: pyiron_base.jobs.job.base.__getitem__ is deprecated: Use job.output for results, job.files to access files; job.content to access HDF storage and job.child_project to access children of master jobs..
  job["output/generic/temperature"]


array([1000.        ,  435.37940295,  582.62354072,  525.90005347,
        668.48802218,  618.6456156 ,  696.48199997,  285.55572155,
        158.06370408,   82.5321783 ,  319.50409436])

# Function 

In [12]:
from pyiron_atomistics import pyiron_to_ase

In [13]:
from pyiron_atomistics.lammps.lammps import lammps_function

In [14]:
output = lammps_function(
    working_directory="test_function",
    structure=pyiron_to_ase(structure),
    potential="1995--Angelo-J-E--Ni-Al-H--LAMMPS--ipr1",
    calc_mode='md',
    calc_kwargs={"pressure": 0.0, "temperature":500.0, "seed":12345},  # some issue with the seed
    cutoff_radius=None,
    units='metal',
    executable_version=None,
    executable_path=None,
    input_control_file=None,
)

In [15]:
output[1]["generic"]["temperature"]

array([1000.        ,  446.27568789,  694.90977856,  474.50744065,
        527.28551136,  696.75148487,  341.39512088,  371.37501195,
        511.99431918,  360.85114921,  728.4821532 ])

# job decorator 

In [16]:
from pyiron_base import job as job_decorator

In [17]:
job = job_decorator(lammps_function)(
    working_directory="test_function",
    structure=pyiron_to_ase(structure),
    potential="1995--Angelo-J-E--Ni-Al-H--LAMMPS--ipr1",
    calc_mode='md',
    calc_kwargs={"pressure": 0.0, "temperature":500.0, "seed":12345},  # some issue with the seed
    cutoff_radius=None,
    units='metal',
    executable_version=None,
    executable_path=None,
    input_control_file=None,
    pyiron_project=pr,
)
job.server.cores

1

In [18]:
job.pull()

The job lammps_function_3ccbd4096461b08b7d4a322173dc81af was saved and received the ID: 2


['LAMMPS (29 Aug 2024)\nOMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)\n  using 1 OpenMP thread(s) per MPI task\nReading data file ...\n  orthogonal box = (0 0 0) to (4.05 4.05 4.05)\n  1 by 1 by 1 MPI processor grid\n  reading atoms ...\n  4 atoms\n  read_data CPU = 0.000 seconds\n\nCITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE\n\nYour simulation uses code contributions which should be cited:\n- Type Label Framework: https://doi.org/10.1021/acs.jpcb.3c08419\nThe log file lists these citations in BibTeX format.\n\nCITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE-CITE\n\nNeighbor list info ...\n  update: every = 1 steps, delay = 0 steps, check = yes\n  max neighbors/atom: 2000, page size: 100000\n  master list distance cutoff = 7.65\n  ghost atom cutoff = 7.65\n  binsize = 3.825, bins = 2 2 2\n  1 neighbor lists, perpetual/occasional/extra = 1 0 0\n  (1) pair eam/alloy, perpetual\n      attributes: half, newton on\n      p

In [19]:
output = job.pull()

In [20]:
output[1]["generic"]["temperature"]

array([1000.        ,  446.27568789,  694.90977856,  474.50744065,
        527.28551136,  696.75148487,  341.39512088,  371.37501195,
        511.99431918,  360.85114921,  728.4821532 ])

In [21]:
df = pr.job_table()
df

,id,status,chemicalformula,job,subjob,projectpath,project,timestart,timestop,totalcputime,computer,hamilton,hamversion,parentid,masterid
0,1,finished,Al4,lmp,/lmp,None,/home/jovyan/test/,2026-04-28 13:47:03.349403,2026-04-28 13:47:05.668512,2.0,pyiron@jupyter-pyiron-dev-pyir-lammps-function-g0os51hv#1,Lammps,0.1,None,None
1,2,finished,None,lammps_function_3ccbd4096461b08b7d4a322173dc81af,/lammps_function_3ccbd4096461b08b7d4a322173dc81af,None,/home/jovyan/test/,2026-04-28 13:47:07.637334,NaT,NaN,pyiron@jupyter-pyiron-dev-pyir-lammps-function-g0os51hv#1,PythonFunctionContainerJob,0.4,None,None


In [22]:
job = pr.load(df[df["job"].str.contains("lammps_function_")]["id"].values[0])

In [23]:
job.output["result"][1]["generic"]["temperature"]

array([1000.        ,  446.27568789,  694.90977856,  474.50744065,
        527.28551136,  696.75148487,  341.39512088,  371.37501195,
        511.99431918,  360.85114921,  728.4821532 ])

# without pyiron_atomitics

In [24]:
from lammpsparser.compatibility.file import lammps_file_interface_function

In [25]:
import os

In [26]:
from ase.build import bulk

In [27]:
shell_output, parsed_output, job_crashed = lammps_file_interface_function(
    working_directory=os.path.abspath("lmp_working_directory"),
    structure=bulk("Al", cubic=True),
    potential="1995--Angelo-J-E--Ni-Al-H--LAMMPS--ipr1",
    calc_mode="md",
    calc_kwargs={"pressure": 0.0, "temperature":500.0, "seed":12345, "n_ionic_steps": 1000},
    units="metal",
)

In [28]:
parsed_output["generic"]["temperature"]

array([1000.        ,  446.27568789,  694.90977856,  474.50744065,
        527.28551136,  696.75148487,  341.39512088,  371.37501195,
        511.99431918,  360.85114921,  728.4821532 ])

In [29]:
job = job_decorator(lammps_file_interface_function)(
    working_directory=os.path.abspath("lmp_working_directory"),
    structure=bulk("Al", cubic=True),
    potential="1995--Angelo-J-E--Ni-Al-H--LAMMPS--ipr1",
    calc_mode="md",
    calc_kwargs={"pressure": 0.0, "temperature":500.0, "seed":12345, "n_ionic_steps": 1000},
    units="metal",
    pyiron_project=pr,
)

In [30]:
output = job.pull()

The job lammps_file_interface_function_f9b23105d8f33cca9d7bc247dfae6fc3 was saved and received the ID: 3


In [31]:
df = pr.job_table()
df

,id,status,chemicalformula,job,subjob,projectpath,project,timestart,timestop,totalcputime,computer,hamilton,hamversion,parentid,masterid
0,1,finished,Al4,lmp,/lmp,None,/home/jovyan/test/,2026-04-28 13:47:03.349403,2026-04-28 13:47:05.668512,2.0,pyiron@jupyter-pyiron-dev-pyir-lammps-function-g0os51hv#1,Lammps,0.1,None,None
1,2,finished,None,lammps_function_3ccbd4096461b08b7d4a322173dc81af,/lammps_function_3ccbd4096461b08b7d4a322173dc81af,None,/home/jovyan/test/,2026-04-28 13:47:07.637334,NaT,NaN,pyiron@jupyter-pyiron-dev-pyir-lammps-function-g0os51hv#1,PythonFunctionContainerJob,0.4,None,None
2,3,finished,None,lammps_file_interface_function_f9b23105d8f33cca9d7bc247dfae6fc3,/lammps_file_interface_function_f9b23105d8f33cca9d7bc247dfae6fc3,None,/home/jovyan/test/,2026-04-28 13:47:12.480440,NaT,NaN,pyiron@jupyter-pyiron-dev-pyir-lammps-function-g0os51hv#1,PythonFunctionContainerJob,0.4,None,None


In [32]:
output[1]["generic"]["temperature"]

array([1000.        ,  446.27568789,  694.90977856,  474.50744065,
        527.28551136,  696.75148487,  341.39512088,  371.37501195,
        511.99431918,  360.85114921,  728.4821532 ])

In [33]:
job = pr.load(df[df["job"].str.contains("lammps_file_interface_function_")]["id"].values[0])

In [34]:
job.output["result"][1]["generic"]["temperature"]

array([1000.        ,  446.27568789,  694.90977856,  474.50744065,
        527.28551136,  696.75148487,  341.39512088,  371.37501195,
        511.99431918,  360.85114921,  728.4821532 ])